# Measuring Gendered Interruptions at the Supreme Court of Canada
## A Computational Analysis of Judicial Power Dynamics in Oral Argument

**Group Members:**
- Sabrin Saide (221178603)
- Matt Aydin (220185328)
- Gobind Dhugee (221173794)

**Course:** Legal Tech Coding — Professor Rehaag, Winter 2026

---

**Research Question:** Can AI-generated transcripts be used to measure whether gendered interruption patterns exist during Supreme Court of Canada oral hearings?


## 1. Setup

First, we clone our project repository from GitHub and install the required Python packages. This only needs to run once.

In [ ]:
import os
import subprocess
import sys

# Clone the repo if we haven't already
if not os.path.exists("scc-interruptions"):
    subprocess.run(["git", "clone", "https://github.com/matt-aydin-2000/scc-interruptions.git"], check=True)
    print("Repository cloned.")
else:
    print("Repository already exists.")

# Install dependencies
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "scc-interruptions/requirements.txt"], check=True)
print("Dependencies installed.")


## 2. Import Our Modules

We add the project directory to Python's path so we can import our custom modules.

In [ ]:
import sys
sys.path.insert(0, "scc-interruptions")
os.chdir("scc-interruptions")

from scraper import fetch_all_transcripts
from transcript_parser import parse_transcript_file
from interruption_detector import detect_interruptions, compute_interruption_metrics, compute_time_to_first_interruption
from analysis import build_justice_dataframe, build_case_level_dataframe, full_analysis, format_results_report
from visualizations import generate_all_visualizations

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use("Agg")
%matplotlib inline

print("All modules imported successfully.")


## 3. Data Acquisition

We scrape 121 AI-generated SCC hearing transcripts from [obiter.ai](https://obiter.ai/scc/), Simon Wallace's project. Wallace transcribed these hearings using OpenAI's Whisper (speech-to-text) and Pyannote (speaker diarization). See: Wallace, S. (2023), "Speaking Like a Judge," *Supreme Court Law Review*, 115(1).

The scraper caches downloaded files, so re-running this cell won't re-download transcripts you already have.

In [ ]:
RAW_DATA_DIR = "data/raw"
PROCESSED_DIR = "data/processed"
OUTPUT_DIR = "output"

os.makedirs(RAW_DATA_DIR, exist_ok=True)
os.makedirs(PROCESSED_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

fetch_all_transcripts(RAW_DATA_DIR, delay=1.0)


## 4. Parse Transcripts into Speaker Turns

Each transcript is HTML with a pattern like `Justice Kasirer (00:09:02): Vous n'allez pas...`. Our parser extracts the speaker name, timestamp, and text from each turn, then tags whether the speaker is a justice or counsel. Justice gender is coded from official SCC pronouns, following Wallace's methodology.

In [ ]:
raw_files = sorted([f for f in os.listdir(RAW_DATA_DIR) if f.endswith(".json")])

cases = []
for filename in raw_files:
    filepath = os.path.join(RAW_DATA_DIR, filename)
    try:
        case = parse_transcript_file(filepath)
        if case["turns"]:
            cases.append(case)
    except Exception as e:
        print(f"  [ERROR] {filename}: {e}")

total_turns = sum(c["total_turns"] for c in cases)
print(f"Parsed {len(cases)} cases with {total_turns} total speaker turns.")

# Show a sample turn so you can see the data structure
print("\nSample speaker turn:")
for k, v in cases[0]["turns"][0].items():
    print(f"  {k}: {v}")


## 5. Detect Interruptions

We detect interruptions using three methods adapted from the US Supreme Court literature (Jacobi & Schweers 2017, Feldman & Gill 2019):

1. **Overlap** — the diarization model explicitly flagged overlapping speech
2. **Timing-based** — a new speaker starts within 15 seconds AND the previous speaker's text looks cut off (ends with a dash, comma, conjunction, etc.)
3. **Rapid judicial intervention** — a justice speaks before counsel has said more than 50 words

We use a 15-second threshold, calibrated through pilot testing.

In [ ]:
all_interruptions = []
all_metrics = []
all_case_data = []
all_ttfi = []

for case in cases:
    turns = case["turns"]
    interruptions = detect_interruptions(turns, time_threshold=15)
    all_interruptions.extend(interruptions)

    for intr in interruptions:
        intr["case_id"] = case["case_id"]
        intr["case_date"] = case["date"]

    metrics = compute_interruption_metrics(interruptions, turns)
    all_metrics.append((case["case_id"], metrics))
    all_case_data.append((case["case_id"], case["date"], metrics, case))

    ttfi = compute_time_to_first_interruption(turns, interruptions)
    all_ttfi.extend(ttfi)

n_overlap = sum(1 for i in all_interruptions if i["type"] == "overlap")
n_timing = sum(1 for i in all_interruptions if i["type"] == "timing")
n_rapid = sum(1 for i in all_interruptions if i["type"] == "rapid_intervention")

print(f"Total interruptions found: {len(all_interruptions)}")
print(f"  Overlapping speech:  {n_overlap}")
print(f"  Timing-based:        {n_timing}")
print(f"  Rapid intervention:  {n_rapid}")


## 6. Justice-Level Summary

We aggregate interruption counts across all cases for each justice, normalized per 1,000 words spoken to control for volubility (Feldman & Gill 2019 found that speaking an additional 1,000 words increases interruption rate by ~40%).

In [ ]:
justice_df = build_justice_dataframe(all_metrics)
case_df = build_case_level_dataframe(all_case_data)

display_cols = ["justice", "gender", "cases_heard", "total_words_spoken",
                "interruptions_made", "interruptions_received",
                "rate_made_per_1k_words", "rate_received_per_1k_words"]
justice_df[display_cols].round(2)


## 7. Statistical Analysis

We run the full battery of tests:
- **Descriptive statistics** (means, medians by gender)
- **Welch's t-tests** and **Mann-Whitney U tests** (comparing male vs female rates)
- **Negative binomial regression** (predicting interruption count from gender, controlling for volubility and Chief Justice status)
- **Cohen's d** effect sizes
- **Z-score outlier analysis**

In [ ]:
results = full_analysis(justice_df, case_df)
report = format_results_report(results)
print(report)


## 8. Visualizations

### Interruption Rates by Gender

In [ ]:
fig_paths = generate_all_visualizations(justice_df, all_interruptions, all_ttfi, OUTPUT_DIR)

from IPython.display import Image, display
for path in fig_paths:
    display(Image(filename=path))


## 9. Key Findings

**Interruptions made:** No statistically significant gender difference. Male justices averaged 2.58 interruptions per 1,000 words spoken; female justices averaged 2.51 (t-test p = 0.92, Mann-Whitney p = 0.48). Cohen's d = 0.057 (negligible effect size).

**Interruptions received:** Male justices appeared to receive more interruptions on average (2.39 vs 1.01 per 1,000 words), but this was driven almost entirely by Chief Justice Wagner (z-score = 2.98), whose procedural role as hearing manager inflates his count. After regression controls, gender was not a significant predictor (p = 0.97).

**Regression:** Neither model (interruptions made or received) found gender to be a significant predictor after controlling for volubility and Chief Justice status.

**Outlier:** Chief Justice Wagner is a strong outlier in both interruptions made and received, consistent with the unique procedural role of the Chief Justice in managing oral hearings.


## 10. COVID-19 Hearing Mode Analysis

The 2020–2022 period covered by our dataset spans a major shift in how SCC hearings were conducted:

| Period | Mode | Details |
|--------|------|---------|
| Jan 2020 – Mar 13, 2020 | **In-person** | Traditional courtroom hearings |
| Mar 14 – Jun 7, 2020 | **Suspended** | No hearings held |
| Jun 8, 2020 – Sep 12, 2021 | **Remote** | Zoom-based hearings |
| Sep 13, 2021 – onward | **Hybrid** | Justices in courtroom, counsel remote |

We classify each hearing by date and compare interruption patterns across modes. This addresses Professor Rehaag's feedback about noting which hearings were conducted remotely.

In [ ]:
from covid_analysis import add_hearing_mode_to_case_df, covid_summary

covid_report, case_df_with_mode = covid_summary(case_df)
print(covid_report)

### COVID Analysis Findings

Interruption rates declined significantly as hearings moved online:

- **In-person** hearings (8 cases) had the highest interruption rate: 2.23 per justice per case
- **Remote/Zoom** hearings (55 cases) dropped to 1.53 (p = 0.019 vs in-person)
- **Hybrid** hearings (58 cases) dropped further to 1.25 (p = 0.036 vs remote)

This suggests that the shift to remote hearings meaningfully reduced interruptions, likely because the turn-taking norms of video conferencing discourage overlapping speech. The gender pattern (males interrupting more) persisted across all hearing modes, though the gap narrowed in hybrid hearings.

We retain all hearing modes in our main analysis but note that the COVID-era shift may attenuate observed interruption rates compared to pre-pandemic norms.

## 11. Validation

Our professor's feedback highlighted validation as the main area for improvement: *"take a look at a random sample of sections of transcripts, manually review for interruptions, then report on accuracy of your automated process."*

We implement two forms of validation:

### A. Manual Validation (Precision & Recall Sampling)

We randomly select 30 detected interruptions and 30 non-flagged speaker changes, display their surrounding transcript context, and assess:
- **Precision**: What proportion of our flagged interruptions are actually real? (Part A)
- **Recall**: What proportion of real interruptions did we catch? (Part B)

In [ ]:
from validation import save_validation_materials

val_report, precision_items, recall_items = save_validation_materials(
    cases, all_interruptions, OUTPUT_DIR, sample_size=30
)

# Show the first 3 precision samples as a preview
print("PREVIEW: First 3 items from the precision check\n")
for i, item in enumerate(precision_items[:3]):
    intr = item["interruption"]
    print(f"--- Sample {i+1} ---")
    print(f"Case: {item['case_id']}")
    print(f"Detection method: {intr['type']}")
    print(f"Interrupter: {intr['interrupter']} ({intr.get('interrupter_gender', '?')})")
    print(f"Interruptee: {intr['interruptee']} ({intr.get('interruptee_gender', '?')})")
    print("Context:")
    for turn in item["context_turns"]:
        marker = " >>>" if turn["turn_index"] == intr["turn_index"] else "    "
        text_preview = turn["text"][:100] + "..." if len(turn["text"]) > 100 else turn["text"]
        print(f"{marker} [{turn['timestamp_str']}] {turn['speaker']}: {text_preview}")
    print()

print(f"Full validation report saved to: output/validation_sample.txt")
print(f"(Contains {len(precision_items)} precision items + {len(recall_items)} recall items)")

### B. LLM Validation (GPT-4o-mini)

We also built a module that sends each flagged interruption's transcript context to OpenAI's GPT-4o-mini and asks the model to:
1. Confirm or reject the interruption classification
2. Classify the type: **hostile**, **clarifying**, **procedural**, or **not an interruption**
3. Provide a brief explanation

This addresses a core weakness of heuristic detection — an LLM can actually understand whether a speaker was cut off mid-thought versus finishing naturally.

**To run LLM validation** (requires an OpenAI API key — costs ~$0.50–2.00 for the full dataset):
```
python llm_validator.py --api-key YOUR_KEY --sample 50
```

We keep this as a separate manual step to control API costs. Results are saved to `output/llm_validation_report.txt`.

## 12. Limitations

- **Interruption detection is approximate.** We use timing heuristics rather than explicit markers (like the double-dash in US SCOTUS transcripts). Our validation sample (Section 11) allows us to quantify this imprecision.
- **Small sample of justices.** Only 10 justices served during this period (6 male, 4 female), limiting statistical power.
- **Chief Justice effect.** Wagner's procedural role distorts the "interruptions received" metric. We control for this in regression.
- **COVID-era remote hearings.** As shown in Section 10, interruption rates dropped significantly when hearings moved online (p = 0.019). We retain all modes in the main analysis but note this as a confound.
- **AI transcription errors.** Wallace's system occasionally misattributes speech during overlapping segments, which both inflates and deflates our counts.
- **No ground-truth benchmark.** Unlike the US SCOTUS (which has professional transcripts with interruption markers), there is no gold-standard Canadian transcript to validate against.

## 13. Save Data

In [ ]:
import json

# Save justice-level metrics
justice_df.to_csv(os.path.join(OUTPUT_DIR, "justice_metrics.csv"), index=False)

# Save case-level data (with hearing mode column)
case_df_with_mode.to_csv(os.path.join(OUTPUT_DIR, "case_level_metrics.csv"), index=False)

# Save interruptions data
with open(os.path.join(PROCESSED_DIR, "all_interruptions.json"), "w") as f:
    json.dump(all_interruptions, f, indent=2, ensure_ascii=False, default=str)

# Save time-to-first-interruption
if all_ttfi:
    pd.DataFrame(all_ttfi).to_csv(os.path.join(OUTPUT_DIR, "time_to_first_interruption.csv"), index=False)

# Save reports
with open(os.path.join(OUTPUT_DIR, "analysis_report.txt"), "w") as f:
    f.write(report)

with open(os.path.join(OUTPUT_DIR, "covid_analysis_report.txt"), "w") as f:
    f.write(covid_report)

print("All output saved to the output/ folder.")